# Limpieza de Datos - Precios de Propiedades en CABA

A partir del dataset crudo de ZonaProp, limpiamos y transformamos
cada columna para dejarla lista para el análisis.

## Pasos
- Limpiar columnas: Convertir strings a números, formateos, etc
- Manejar nulos
- Detectar y eliminar outliers
- Exportar dataset limpio

### 1. Imports y carga

In [53]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("../data/raw/zonaprop_raw.csv")
print(f"Dataset crudo: {df_raw.shape}")
df = df_raw.copy()
df.head()

Dataset crudo: (38298, 7)


,precio,expensas,metros,ambientes,banos,direccion,barrio
0,USD 530.000,$ 770.000 Expensas,172 m² tot.,4 amb.,3 baños,Soldado De La Independencia 1200,"Las Cañitas, Palermo"
1,USD 170.000,NaN,73 m² tot.,4 amb.,1 baño,Malabia al 200,"Villa Crespo, Capital Federal"
2,USD 120.000,$ 260.000 Expensas,54 m² tot.,3 amb.,1 baño,Paraguay e/ Jean Jaures y Doctor Tomás Manuel ...,"Recoleta, Capital Federal"
3,USD 220.000,$ 380.073 Expensas,68 m² tot.,3 amb.,2 baños,PICO 1600. Entre Libertador y 11 De Septiembre,"Núñez, Capital Federal"
4,USD 2.250.000,$ 2.100.000 Expensas,399 m² tot.,4 amb.,3 baños,Av. Alvear 1500,"Recoleta, Capital Federal"


### 2. Limpieza

#### 2.1 Limpieza columna precios

Primero veo si todos los precios estan en el mismo formato "USD X.XXX"

In [54]:
raros = df["precio"][~df["precio"].str.startswith("USD")].value_counts()
print(f"Valores fuera del formato USD: {len(raros)} tipos distintos")
print(raros)

Valores fuera del formato USD: 4 tipos distintos
precio
Consultar precio    175
$ 265.000             1
$ 110.000             1
$ 142.000             1
Name: count, dtype: int64


Hay 175 propiedades sin precio y otras 3 con el precio es Pesos Argentinos.
Son solo 178 filas sobre 35k, menos del 0.5%.

In [ ]:
def limpiar_precio(valor):
    if pd.isna(valor):
        return None
    if not valor.startswith("USD"):
        return None
    try:
        valor = valor.replace("USD ", "").replace(".", "")
        return int(valor)
    except ValueError:
        return None

df["precio"] = df["precio"].apply(limpiar_precio)

print(f"Nulos: {df['precio'].isnull().sum()}")
print(f"\nMin: USD {df['precio'].min():,}")
print(f"Max: USD {df['precio'].max():,}")
print(f"Media: USD {df['precio'].mean():,.0f}")

Nulos: 178

Min: USD 1.0
Max: USD 50,520,900.0
Media: USD 400,594


#### 2.2 Limpieza columna expensas

Primero veo si todas las exprensas estan en el mismo formato "$ XXX.XXX Expensas"

In [56]:
raros_exp = df["expensas"][
    df["expensas"].notna() & 
    ~df["expensas"].str.startswith("$")
].value_counts()

print(f"Valores fuera del formato: {len(raros_exp)} tipos distintos")
print(raros_exp)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


Todas las expensas estan en el mismo formato

In [57]:
def limpiar_expensas(valor):
    if pd.isna(valor):
        return 0  # sin expensas = 0
    try:
        valor = valor.replace("$ ", "").replace(" Expensas", "").replace(".", "")
        return int(valor)
    except ValueError:
        return 0

df["expensas"] = df["expensas"].apply(limpiar_expensas)

print(f"Nulos: {df['expensas'].isnull().sum()}")
print(f"Ceros (sin expensas): {(df['expensas'] == 0).sum()}")
print(f"\nMin: $ {df['expensas'].min():,}")
print(f"Max: $ {df['expensas'].max():,}")
print(f"Media: $ {df['expensas'].mean():,.0f}")

Nulos: 0
Ceros (sin expensas): 16001

Min: $ 0
Max: $ 48,090,200
Media: $ 226,074


#### 2.3 Limpieza metros colummna

Veo si todos los valores en la columna siguen el mismo formato "Xm² tot."

In [58]:
raros_metros = df["metros"][
    df["metros"].notna() &
    ~df["metros"].str.contains("m²")
].value_counts()

print(f"Valores fuera del formato: {len(raros_metros)} tipos distintos")
print(raros_metros)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


Hago el formateo y ademas elimino los valores nulos (18 en total).

In [59]:
def limpiar_metros(valor):
    if pd.isna(valor):
        return None
    try:
        valor = valor.replace(" m² tot.", "").replace(".", "")
        return int(valor)
    except ValueError:
        return None

df["metros"] = df["metros"].apply(limpiar_metros)

# Eliminar los 18 nulos
df = df.dropna(subset=["metros"])

print(f"Filas después de limpiar metros: {len(df)}")
print(f"\nMin: {df['metros'].min()} m²")
print(f"Max: {df['metros'].max()} m²")
print(f"Media: {df['metros'].mean():.0f} m²")

Filas después de limpiar metros: 38280

Min: 1.0 m²
Max: 3500000.0 m²
Media: 473 m²


#### 2.4 Limpieza columna ambientes

Veo que todas las entradas tengan el mismo formato "X amb."

In [60]:
raros_amb = df["ambientes"][
    df["ambientes"].notna() &
    ~df["ambientes"].str.contains("amb")
].value_counts()

print(f"Valores fuera del formato: {len(raros_amb)} tipos distintos")
print(raros_amb)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


Formateo:

In [61]:
def limpiar_ambientes(valor):
    if pd.isna(valor):
        return None
    try:
        valor = valor.replace(" amb.", "")
        return int(valor)
    except ValueError:
        return None

df["ambientes"] = df["ambientes"].apply(limpiar_ambientes)

print(f"Nulos: {df['ambientes'].isnull().sum()}")
print(f"\nMin: {df['ambientes'].min()}")
print(f"Max: {df['ambientes'].max()}")
print(f"Media: {df['ambientes'].mean():.1f}")

Nulos: 3269

Min: 1.0
Max: 56.0
Media: 3.1


### 2.5 Limpieza columna banos

In [62]:
raros_banos = df["banos"][
    df["banos"].notna() &
    ~df["banos"].str.contains("baño")
].value_counts()

print(f"Valores fuera del formato: {len(raros_banos)} tipos distintos")
print(raros_banos)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


In [63]:
def limpiar_banos(valor):
    if pd.isna(valor):
        return None
    try:
        valor = valor.replace(" baños", "").replace(" baño", "")
        return int(valor)
    except ValueError:
        return None

df["banos"] = df["banos"].apply(limpiar_banos)

print(f"Nulos: {df['banos'].isnull().sum()}")
print(f"\nMin: {df['banos'].min()}")
print(f"Max: {df['banos'].max()}")
print(f"Media: {df['banos'].mean():.1f}")

Nulos: 2637

Min: 1.0
Max: 70.0
Media: 1.7


#### 2.6 Limpieza columna barrio

Primero veo en que formatos viene la columna barrio

In [64]:
print("Ejemplos de barrio:")
print(df["barrio"].head(15))

print(f"\nBarrios únicos: {df['barrio'].nunique()}")
print("\nMás frecuentes:")
print(df["barrio"].value_counts().head())

Ejemplos de barrio:
0                      Las Cañitas, Palermo
1             Villa Crespo, Capital Federal
2                 Recoleta, Capital Federal
3                    Núñez, Capital Federal
4                 Recoleta, Capital Federal
5               Villa Luro, Capital Federal
6     Centro / Microcentro, Capital Federal
7                  Liniers, Capital Federal
8                  Almagro, Capital Federal
9                 Belgrano, Capital Federal
10               Chacarita, Capital Federal
11        Villa del Parque, Capital Federal
12                 Caballito Sur, Caballito
13                Recoleta, Capital Federal
14                Belgrano, Capital Federal
Name: barrio, dtype: str

Barrios únicos: 83

Más frecuentes:
barrio
Recoleta, Capital Federal     3717
Belgrano, Capital Federal     3429
Palermo, Capital Federal      2743
Caballito, Capital Federal    1916
Núñez, Capital Federal        1408
Name: count, dtype: int64


La columna barrio viene en formato `"X, Y"` donde a veces X es el barrio 
y otras veces Y es el barrio, dependiendo de si se trata de un sub-barrio o 
barrio principal. Por ejemplo:

- `"Recoleta, Capital Federal"` → el barrio es Recoleta
- `"Las Cañitas, Palermo"` → el barrio es Palermo
- `"Caballito Sur, Caballito"` → el barrio es Caballito

Para estandarizar esta columna voy a utilizar el dataset oficial de barrios 
de la Ciudad de Buenos Aires (GCBA), que contiene los 48 barrios oficiales 
y la comuna a la que pertenece cada uno. De esta forma no solo limpio y 
estandarizo los nombres, sino que además agrego la columna `comuna` como 
feature adicional.

In [65]:
barrios = pd.read_csv("../data/raw/barrios_comunas_p_Ciencia_de_Datos_y_PP.csv")
print(f"Shape barrios dataset: {barrios.shape}")
print(f"Columnas de barrios dataset: {barrios.columns.tolist()}")
barrios.head()

Shape barrios dataset: (48, 2)
Columnas de barrios dataset: ['BARRIO', 'COMUNA']


,BARRIO,COMUNA
0,AGRONOMIA,15
1,ALMAGRO,5
2,BALVANERA,3
3,BARRACAS,4
4,BELGRANO,13


Para determinar la lógica de limpieza, extraigo ambas partes del formato "X, Y" de cada valor, las normalizo a mayúsculas y las comparo contra los 
48 barrios oficiales del GCBA. Esto me permite saber cuántos barrios matchean por cada lado antes de definir la función de limpieza.

In [66]:
izquierda = df["barrio"].str.split(",").str[0].str.strip().str.upper().unique()
derecha = df["barrio"].str.split(",").str[1].str.strip().str.upper().unique()

barrios_gcba = barrios["BARRIO"].unique()

match_izq = [b for b in izquierda if b in barrios_gcba]
match_der = [b for b in derecha if b in barrios_gcba]

print(f"Matchean por izquierda: {len(match_izq)}")
print(f"Matchean por derecha: {len(match_der)}")

Matchean por izquierda: 39
Matchean por derecha: 9


Identifico cuántas filas representa cada barrio que no matcheó con el dataset oficial. Para decidir si descartarlos o 
mapearlos manualmente.

In [67]:
no_matchean = []

for valor in df["barrio"].unique():
    partes = valor.split(",")
    izq = partes[0].strip().upper()
    der = partes[1].strip().upper() if len(partes) > 1 else ""
    
    if izq not in barrios_gcba and der not in barrios_gcba:
        no_matchean.append(valor)

print(f"Sin match en ningún lado: {len(no_matchean)}")
print(sorted(no_matchean))

Sin match en ningún lado: 17
['Abasto, Capital Federal', 'Agronomía, Capital Federal', 'Barrio Norte, Capital Federal', 'Centro / Microcentro, Capital Federal', 'Congreso, Capital Federal', 'Constitución, Capital Federal', 'La Boca, Capital Federal', 'La Paternal, Capital Federal', 'Lomas de Núñez, Núñez', 'Núñez, Capital Federal', 'Once, Capital Federal', 'Parque Centenario, Capital Federal', 'Pompeya, Capital Federal', 'San Nicolás, Capital Federal', 'Tribunales, Capital Federal', 'Villa General Mitre, Capital Federal', 'Villa Pueyrredón, Capital Federal']


In [68]:
for valor in sorted(no_matchean):
    count = (df["barrio"] == valor).sum()
    print(f"{count:4d} filas → {valor}")

  77 filas → Abasto, Capital Federal
  91 filas → Agronomía, Capital Federal
1015 filas → Barrio Norte, Capital Federal
 438 filas → Centro / Microcentro, Capital Federal
  97 filas → Congreso, Capital Federal
 230 filas → Constitución, Capital Federal
  94 filas → La Boca, Capital Federal
 243 filas → La Paternal, Capital Federal
   8 filas → Lomas de Núñez, Núñez
1408 filas → Núñez, Capital Federal
  70 filas → Once, Capital Federal
  42 filas → Parque Centenario, Capital Federal
  79 filas → Pompeya, Capital Federal
 491 filas → San Nicolás, Capital Federal
  38 filas → Tribunales, Capital Federal
 103 filas → Villa General Mitre, Capital Federal
 251 filas → Villa Pueyrredón, Capital Federal


Son demasiadas filas para eliminar, por ejemplo, Barrio Norte (1014 filas) y 
Núñez (1377 filas) son demasiado importantes para perder.

Para los 17 barrios sin match creo un diccionario de mapeo manual que los traduce a su nombre oficial en el GCBA. Luego defino la función 
`limpiar_barrio` que aplica este mapeo, busca el match en ambos lados del formato "X, Y" y retorna el barrio oficial junto con su comuna.

In [69]:
mapeo_manual = {"ABASTO": "BALVANERA", "AGRONOMÍA": "AGRONOMIA", "BARRIO NORTE": "RECOLETA",
    "CENTRO / MICROCENTRO": "SAN NICOLAS", "CONGRESO": "BALVANERA", "CONSTITUCIÓN": "CONSTITUCION",
    "LA BOCA": "BOCA", "LA PATERNAL": "PATERNAL", "LOMAS DE NÚÑEZ": "NUÑEZ",
    "NÚÑEZ": "NUÑEZ", "ONCE": "BALVANERA", "PARQUE CENTENARIO": "CABALLITO",
    "POMPEYA": "NUEVA POMPEYA", "SAN NICOLÁS": "SAN NICOLAS",
    "TRIBUNALES": "SAN NICOLAS", "VILLA GENERAL MITRE": "VILLA GRAL. MITRE",
    "VILLA PUEYRREDÓN": "VILLA PUEYRREDON"
}

In [70]:
def limpiar_barrio(valor, barrios_gcba, mapeo_manual):
    if pd.isna(valor):
        return None, None
    
    partes = valor.split(",")
    izq = partes[0].strip().upper()
    der = partes[1].strip().upper() if len(partes) > 1 else ""
    
    # Aplicar mapeo manual primero
    if izq in mapeo_manual:
        izq = mapeo_manual[izq]
    if der in mapeo_manual:
        der = mapeo_manual[der]
    
    # Buscar match con barrios oficiales
    if izq in barrios_gcba:
        barrio = izq
    elif der in barrios_gcba:
        barrio = der
    else:
        return None, None
    
    # Obtener comuna
    comuna = barrios.loc[barrios["BARRIO"] == barrio, "COMUNA"].values[0]
    
    return barrio, comuna

# Aplicar
barrios_gcba_set = set(barrios["BARRIO"].unique())

df[["barrio_clean", "comuna"]] = df["barrio"].apply(
    lambda x: pd.Series(limpiar_barrio(x, barrios_gcba_set, mapeo_manual))
)

print(f"Nulos en barrio_clean: {df['barrio_clean'].isnull().sum()}")
print(f"Nulos en comuna: {df['comuna'].isnull().sum()}")
print(f"\nBarrios únicos: {df['barrio_clean'].nunique()}")
print(f"\nDistribución por comuna:")
print(df["comuna"].value_counts().sort_index())

Nulos en barrio_clean: 0
Nulos en comuna: 0

Barrios únicos: 48

Distribución por comuna:
comuna
1     4235
2     4732
3     1614
4      901
5     1614
6     2847
7     1275
8      145
9      662
10     779
11    1902
12    2588
13    6584
14    6526
15    1876
Name: count, dtype: int64


### 3. Manejo de nulos

In [71]:
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(2)

pd.DataFrame({
    "nulos": nulos,
    "porcentaje %": porcentaje
})

,nulos,porcentaje %
precio,178,0.46
expensas,0,0.00
metros,0,0.00
ambientes,3269,8.54
banos,2637,6.89
direccion,0,0.00
barrio,0,0.00
barrio_clean,0,0.00
comuna,0,0.00


#### 3.1 Nulos en precio

Los 178 valores nulos en precio corresponden a propiedades con "Consultar precio" o precios en Pesos Argentinos.
Representan menos del 0.5% del dataset por lo que al eliminarlas no hay pérdida significativa.

In [72]:
df = df.dropna(subset=["precio"])
print(f"Filas después de eliminar nulos en precio: {len(df)}")

Filas después de eliminar nulos en precio: 38102


#### 3.2 Nulos en ambientes y baños

¿Cuánto se solapan los nulos de ambientes y baños?

In [73]:
ambos = df["ambientes"].isnull() & df["banos"].isnull()
solo_amb = df["ambientes"].isnull() & df["banos"].notna()
solo_ban = df["banos"].isnull() & df["ambientes"].notna()

print(f"Nulos en ambos:          {ambos.sum()}")
print(f"Nulos solo en ambientes: {solo_amb.sum()}")
print(f"Nulos solo en baños:     {solo_ban.sum()}")
print(f"Total filas afectadas:   {(ambos | solo_amb | solo_ban).sum()}")

Nulos en ambos:          2036
Nulos solo en ambientes: 1181
Nulos solo en baños:     589
Total filas afectadas:   3806


Dado que el dataset cuenta con ~38k propiedades, eliminar estas filas nos deja con ~34k, 
suficiente para el análisis y quizas un modelado posterior.

In [74]:
df = df.dropna(subset=["ambientes", "banos"])

print(f"Filas después de eliminar nulos: {len(df)}")
print(f"\nNulos restantes:")
print(df[["precio", "expensas", "metros", "ambientes", "banos", "barrio_clean"]].isnull().sum())

Filas después de eliminar nulos: 34296

Nulos restantes:
precio          0
expensas        0
metros          0
ambientes       0
banos           0
barrio_clean    0
dtype: int64


### 4. Detección y eliminación de outliers

Los valores extremos detectados durante la limpieza indican la presencia de outliers, hay propiedades con precios, superficies o características 
imposibles o poco representativas del mercado inmobiliario de CABA. Los eliminamos columna por columna analizando cada distribución.

#### 4.1 Columna precio

In [75]:
print("PRECIO")
print(f"Min:    USD {df['precio'].min():,}")
print(f"Max:    USD {df['precio'].max():,}")
print(f"Media:  USD {df['precio'].mean():,.0f}")
print(f"Mediana: USD {df['precio'].median():,.0f}")
print(f"\nDistribución por percentiles:")
percentiles = [1, 5, 25, 50, 75, 95, 99]
for p in percentiles:
    print(f"  p{p:02d}: USD {df['precio'].quantile(p/100):,.0f}")

PRECIO
Min:    USD 1.0
Max:    USD 12,400,000.0
Media:  USD 325,698
Mediana: USD 189,500

Distribución por percentiles:
  p01: USD 54,900
  p05: USD 70,000
  p25: USD 122,021
  p50: USD 189,500
  p75: USD 339,000
  p95: USD 890,000
  p99: USD 2,500,000


Utilizo los percentiles 1 y 99 como límites para eliminar outliers en precio. Esto significa conservar el 98% central de los datos, eliminando 
el 1% más barato y el 1% más caro.

In [76]:
p01 = df["precio"].quantile(0.01)
p99 = df["precio"].quantile(0.99)

df = df[(df["precio"] >= p01) & (df["precio"] <= p99)]

print(f"Límite inferior: USD {p01:,.0f}")
print(f"Límite superior: USD {p99:,.0f}")
print(f"Filas después de filtrar: {len(df)}")

Límite inferior: USD 54,900
Límite superior: USD 2,500,000
Filas después de filtrar: 33670


#### 4.2 Outliers en metros

In [77]:
print("METROS")
print(f"Min:    {df['metros'].min()} m²")
print(f"Max:    {df['metros'].max()} m²")
print(f"Media:  {df['metros'].mean():.0f} m²")
print(f"Mediana: {df['metros'].median():.0f} m²")
print(f"\nDistribución por percentiles:")
percentiles = [1, 5, 25, 50, 75, 95, 99]
for p in percentiles:
    print(f"  p{p:02d}: {df['metros'].quantile(p/100):.0f} m²")

METROS
Min:    1.0 m²
Max:    3500000.0 m²
Media:  468 m²
Mediana: 78 m²

Distribución por percentiles:
  p01: 26 m²
  p05: 34 m²
  p25: 49 m²
  p50: 78 m²
  p75: 132 m²
  p95: 294 m²
  p99: 488 m²


Aplico límites manuales basados en el contexto del mercado inmobiliario de CABA y aproximados
a los percentiles 01 y 99. El mínimo de 25 m² corresponde al tamaño mínimo real de un 
monoambiente, y el máximo de 500 m² a una casa grande. Valores fuera 
de este rango son errores de carga o propiedades no representativas.

In [78]:
df = df[(df["metros"] >= 25) & (df["metros"] <= 500)]

print(f"Límite inferior: 25 m²")
print(f"Límite superior: 500 m²")
print(f"Filas después de filtrar: {len(df)}")

Límite inferior: 25 m²
Límite superior: 500 m²
Filas después de filtrar: 33265


#### 4.3 Outliers en ambientes

In [79]:
print("AMBIENTES")
print(f"Min:    {df['ambientes'].min()}")
print(f"Max:    {df['ambientes'].max()}")
print(f"Media:  {df['ambientes'].mean():.1f}")
print(f"Mediana: {df['ambientes'].median():.0f}")
print(f"\nDistribución por percentiles:")
percentiles = [1, 5, 25, 50, 75, 95, 99]
for p in percentiles:
    print(f"  p{p:02d}: {df['ambientes'].quantile(p/100):.0f}")

print(f"\nDistribución de valores:")
print(df["ambientes"].value_counts().sort_index())

AMBIENTES
Min:    1.0
Max:    22.0
Media:  3.0
Mediana: 3

Distribución por percentiles:
  p01: 1
  p05: 1
  p25: 2
  p50: 3
  p75: 4
  p95: 5
  p99: 8

Distribución de valores:
ambientes
1.0     5650
2.0     7413
3.0     9061
4.0     7181
5.0     2414
6.0      793
7.0      354
8.0      215
9.0      136
10.0      25
11.0       8
12.0       5
13.0       4
14.0       1
16.0       1
17.0       1
21.0       2
22.0       1
Name: count, dtype: int64


Dado que es una variable discreta, analizo la distribución completa. 
A partir de 10 ambientes los casos son insignificantes, los elimino.

In [80]:
df = df[df["ambientes"] <= 10]
print(f"Filas después de filtrar: {len(df)}")

Filas después de filtrar: 33242


#### 4.4 Outliers de baños

In [81]:
print("BAÑOS")
print(f"Min:    {df['banos'].min()}")
print(f"Max:    {df['banos'].max()}")
print(f"\nDistribución de valores:")
print(df["banos"].value_counts().sort_index())

BAÑOS
Min:    1.0
Max:    20.0

Distribución de valores:
banos
1.0     19124
2.0      9739
3.0      3248
4.0       956
5.0       143
6.0        25
7.0         4
10.0        2
20.0        1
Name: count, dtype: int64


Igual que ambientes, analizo la distribución completa por ser variable 
discreta. A partir de 6 baños los casos son insignificantes, los elimino.

In [82]:
df = df[df["banos"] <= 6]
print(f"Filas después de filtrar: {len(df)}")

Filas después de filtrar: 33235


#### 4.5 Outliers de expensas

In [83]:
print("EXPENSAS")
print(f"Min:    $ {df['expensas'].min():,}")
print(f"Max:    $ {df['expensas'].max():,}")
print(f"Media:  $ {df['expensas'].mean():,.0f}")
print(f"Mediana: $ {df['expensas'].median():,.0f}")
print(f"\nDistribución por percentiles:")
percentiles = [1, 5, 25, 50, 75, 95, 99]
for p in percentiles:
    print(f"  p{p:02d}: $ {df['expensas'].quantile(p/100):,.0f}")

EXPENSAS
Min:    $ 0
Max:    $ 48,090,200
Media:  $ 215,825
Mediana: $ 118,000

Distribución por percentiles:
  p01: $ 0
  p05: $ 0
  p25: $ 0
  p50: $ 118,000
  p75: $ 280,000
  p95: $ 820,000
  p99: $ 1,500,000


El mínimo de 0 es válido, corresponde a propiedades sin expensas. 
Para el máximo uso p99 ($1.500.000).

In [84]:
p99_exp = df["expensas"].quantile(0.99)
df = df[df["expensas"] <= p99_exp]

print(f"Límite superior: $ {p99_exp:,.0f}")
print(f"Filas después de filtrar: {len(df)}")

Límite superior: $ 1,500,000
Filas después de filtrar: 32946


Resumen:

In [86]:
print("RESUMEN FINAL")
print("="*40)
print(f"\nDataset crudo:   {len(df_raw):,} filas")
print(f"Dataset limpio:  {len(df):,} filas")
print(f"Filas perdidas:  {len(df_raw) - len(df):,} ({(len(df_raw) - len(df)) / len(df_raw) * 100:.1f}%)")
print(f"\nColumnas finales:")
print(df.columns.tolist())
print(f"\nNulos:")
print(df.isnull().sum())
print(f"\nEstadísticas finales:")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print(df[["precio", "metros", "ambientes", "banos", "expensas"]].describe())

RESUMEN FINAL

Dataset crudo:   38,298 filas
Dataset limpio:  32,946 filas
Filas perdidas:  5,352 (14.0%)

Columnas finales:
['precio', 'expensas', 'metros', 'ambientes', 'banos', 'direccion', 'barrio', 'barrio_clean', 'comuna']

Nulos:
precio          0
expensas        0
metros          0
ambientes       0
banos           0
direccion       0
barrio          0
barrio_clean    0
comuna          0
dtype: int64

Estadísticas finales:
            precio    metros  ambientes     banos     expensas
count    32,946.00 32,946.00  32,946.00 32,946.00    32,946.00
mean    275,377.41    103.85       2.95      1.58   194,138.14
std     256,847.79     80.11       1.44      0.81   266,719.43
min      54,900.00     25.00       1.00      1.00         0.00
25%     123,000.00     49.00       2.00      1.00         0.00
50%     187,000.00     77.00       3.00      1.00   113,000.00
75%     320,000.00    128.00       4.00      2.00   270,000.00
max   2,500,000.00    500.00      10.00      6.00 1,500,000.0

### 5. Export

In [87]:
# Sacar columnas que no voy a usar
df_final = df.drop(columns=["direccion", "barrio"])

# Renombrar barrio_clean a barrio
df_final = df_final.rename(columns={"barrio_clean": "barrio"})

print(df_final.columns.tolist())
df_final.head()

['precio', 'expensas', 'metros', 'ambientes', 'banos', 'barrio', 'comuna']


,precio,expensas,metros,ambientes,banos,barrio,comuna
0,"530,000.00",770000,172.00,4.00,3.00,PALERMO,14
1,"170,000.00",0,73.00,4.00,1.00,VILLA CRESPO,15
2,"120,000.00",260000,54.00,3.00,1.00,RECOLETA,2
3,"220,000.00",380073,68.00,3.00,2.00,NUÑEZ,13
5,"84,100.00",110000,55.00,1.00,1.00,VILLA LURO,10


In [88]:
df_final.to_csv("../data/processed/zonaprop_clean.csv", index=False)

print(f"Dataset exportado exitosamente")
print(f"Filas: {len(df_final)}")
print(f"Columnas: {df_final.columns.tolist()}")

Dataset exportado exitosamente
Filas: 32946
Columnas: ['precio', 'expensas', 'metros', 'ambientes', 'banos', 'barrio', 'comuna']
